<a href="https://colab.research.google.com/github/gilIolgenblum/CrowdingModeWorkshop/blob/main/tutorials/solutions/E1_forward_simulation_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 1 — Solution: Forward Simulation

> ⚠️ **This is the solution notebook.** Students should attempt the exercises independently first.

In [ ]:
# (Only needed on Colab — skip if running locally)
# !pip install git+https://github.com/gilIolgenblum/CrowdingModeWorkshop.git

In [ ]:
import crowding as cr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

---
## Task A1 — Build a Model from Scratch

In [ ]:
# Define Protein
MET16 = cr.Protein(SASA=419.0)

# Define TMAO Cosolute
tmao = cr.Cosolute(nu=3.98, chi=-0.74, chiTS=-6.0)

# Build and solve the model
model_tmao = cr.CrowdingModel(
    protein=MET16,
    cosolute=tmao,
    eps=0.0, epsTS=0.0,
    phiC_max=0.20, dphiC=0.001
)
model_tmao.solve()

# Extract ΔΔG⁰ at φ = 0.10
idx = np.argmin(np.abs(model_tmao.phiC - 0.10))
ddG_at_010 = model_tmao.ddA_kj[idx]
print(f"ΔΔG⁰ at φ=0.10: {ddG_at_010:.4f} kJ/mol")

---
## Task A2 — Compare Two Cosolutes

In [ ]:
# Glycerol cosolute
glycerol = cr.Cosolute(nu=2.479, chi=0.610, chiTS=-3.650)
model_glycerol = cr.CrowdingModel(
    protein=MET16, cosolute=glycerol,
    eps=0.0, epsTS=0.0,
    phiC_max=0.20, dphiC=0.001
)
model_glycerol.solve()

# Plot comparison
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(model_tmao.phiC, model_tmao.ddA_kj, label="TMAO")
ax.plot(model_glycerol.phiC, model_glycerol.ddA_kj, label="Glycerol")
ax.set_xlabel(r"$\phi_C$")
ax.set_ylabel(r"$\Delta\Delta G^0$ [kJ/mol]")
ax.legend()
ax.set_title("MET16: TMAO vs. Glycerol")
plt.tight_layout()
plt.show()

# At phi=0.10
idx = np.argmin(np.abs(model_tmao.phiC - 0.10))
print(f"TMAO    ΔΔG⁰ at φ=0.10: {model_tmao.ddA_kj[idx]:.4f} kJ/mol")
print(f"Glycerol ΔΔG⁰ at φ=0.10: {model_glycerol.ddA_kj[idx]:.4f} kJ/mol")

**Answer:** Glycerol destabilizes MET16 more than TMAO at φ=0.10 (more negative/less positive ΔΔG⁰),
because its larger chi parameter introduces a strong enthalpic destabilization.

---
## Task A3 — Sensitivity to ν

In [ ]:
# Glycerol with doubled nu
glycerol_2nu = cr.Cosolute(nu=5.0, chi=0.610, chiTS=-3.650)
model_glycerol_2nu = cr.CrowdingModel(
    protein=MET16, cosolute=glycerol_2nu,
    eps=0.0, epsTS=0.0,
    phiC_max=0.20, dphiC=0.001
)
model_glycerol_2nu.solve()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# ΔΔG⁰
axes[0].plot(model_glycerol.phiC, model_glycerol.ddA_kj, label=r"$\nu=2.479$")
axes[0].plot(model_glycerol_2nu.phiC, model_glycerol_2nu.ddA_kj, label=r"$\nu=5.0$")
axes[0].set_xlabel(r"$\phi_C$")
axes[0].set_ylabel(r"$\Delta\Delta G^0$ [kJ/mol]")
axes[0].legend()
axes[0].set_title(r"$\Delta\Delta G^0$")

# TΔΔSi⁰
axes[1].plot(model_glycerol.phiC, model_glycerol.TddS_kj, label=r"$\nu=2.479$")
axes[1].plot(model_glycerol_2nu.phiC, model_glycerol_2nu.TddS_kj, label=r"$\nu=5.0$")
axes[1].set_xlabel(r"$\phi_C$")
axes[1].set_ylabel(r"$T\Delta\Delta S^0$ [kJ/mol]")
axes[1].legend()
axes[1].set_title(r"$T\Delta\Delta S^0$")

plt.tight_layout()
plt.show()

**Answer:** ΔΔH⁰ does **not** change when ν is doubled.  
In the Flory-Huggins model, ν controls the excluded volume (purely entropic) term.  
The enthalpy is governed by χ and ε only. Doubling ν affects only TΔΔSi⁰.

---
## Task A4 — Add a Soft Interaction

In [ ]:
# Glycerol model with soft interaction
model_glycerol_eps = cr.CrowdingModel(
    protein=MET16, cosolute=glycerol,
    eps=-0.5, epsTS=0.3,
    phiC_max=0.20, dphiC=0.001
)
model_glycerol_eps.solve()

plotter = cr.Plotter(model_glycerol_eps)
fig = plotter.plot_all()
plt.show()

**Answer:** Adding `eps=-0.5` (with `epsTS=0.3`, so `epsH = eps - epsTS = -0.8`) shifts **ΔΔH⁰** 
to more negative values — it adds an attractive enthalpic contribution.
TΔΔSi⁰ is also shifted by the entropic component `epsTS`.
The soft interaction thus modifies **both** the enthalpic and entropic channels.

---
## Task A5 — Export Results to CSV

In [ ]:
df = model_glycerol_eps.to_pandas()
print(df.head())
df.to_csv("my_results.csv", index=False)
print("Saved to my_results.csv")